# M6 round 1 — LLaVA-1.5-7B cross-model H2 + H1 + H7

Reproduces the headline tables from `docs/insights/m6_cross_model_llava.md`:
- Paired PMR delta vs label-free for both Qwen2.5-VL and LLaVA-1.5
- Cross-model PMR S-curve over `object_level`
- H7 GAR-by-label cross-model gradient
- The `line/blank/none × label` 4-row breakdown for each model

Depends on prior outputs:
- `outputs/mvp_full_20260424-094103_8ae1fa3d/` (M2 — Qwen labeled)
- `outputs/label_free_20260425-031430_315c5318/` (M4b — Qwen label-free)
- `outputs/cross_model_llava_20260425-035506_7ff0256b/` (M6 — LLaVA labeled)
- `outputs/cross_model_llava_label_free_20260425-040821_39e68cd4/` (M6 — LLaVA label-free)

If the LLaVA runs haven't been executed yet:
```bash
uv run python scripts/02_run_inference.py --config configs/cross_model_llava.py            --stimulus-dir inputs/mvp_full_20260424-093926_e9d79da3
uv run python scripts/02_run_inference.py --config configs/cross_model_llava_label_free.py --stimulus-dir inputs/mvp_full_20260424-093926_e9d79da3
uv run python scripts/03_score_and_summarize.py --run-dir outputs/cross_model_llava_<ts>
uv run python scripts/03_score_and_summarize.py --run-dir outputs/cross_model_llava_label_free_<ts>
```
(~40 min total on H200.)

In [1]:
import sys
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

QWEN_LBL  = PROJECT_ROOT / 'outputs' / 'mvp_full_20260424-094103_8ae1fa3d'      # M2
QWEN_NL   = PROJECT_ROOT / 'outputs' / 'label_free_20260425-031430_315c5318'    # M4b
LLAVA_LBL = PROJECT_ROOT / 'outputs' / 'cross_model_llava_20260425-035506_7ff0256b'
LLAVA_NL  = PROJECT_ROOT / 'outputs' / 'cross_model_llava_label_free_20260425-040821_39e68cd4'

Q_LBL  = pd.read_parquet(QWEN_LBL / 'predictions_scored.parquet')
Q_LBL  = Q_LBL[Q_LBL.prompt_variant == 'open']  # M6 ran open only on LLaVA
Q_NL   = pd.read_parquet(QWEN_NL / 'predictions_scored.parquet')
L_LBL  = pd.read_parquet(LLAVA_LBL / 'predictions_scored.parquet')
L_NL   = pd.read_parquet(LLAVA_NL / 'predictions_scored.parquet')

print(f'Qwen  open n={len(Q_LBL)}, label-free n={len(Q_NL)}')
print(f'LLaVA open n={len(L_LBL)}, label-free n={len(L_NL)}')

Qwen  open n=1440, label-free n=480
LLaVA open n=1440, label-free n=480


## Headline: paired PMR delta `PMR(label) − PMR(_nolabel)`

Same-image, same-seed pairing on `sample_id`. Qwen and LLaVA show
opposite-direction signals for `circle`, and very different magnitudes
for `ball` and `planet`.

In [2]:
def paired_delta(labeled, label_free):
    nl = label_free.groupby('sample_id').pmr.mean().rename('_nolabel')
    rows = []
    for lab in ['ball', 'circle', 'planet']:
        lbl = labeled[labeled.label == lab].groupby('sample_id').pmr.mean().rename(lab)
        j = pd.concat([nl, lbl], axis=1).dropna()
        rows.append({'label': lab, 'mean_delta': float((j[lab] - j._nolabel).mean()), 'n': len(j)})
    return pd.DataFrame(rows).set_index('label')

qwen_delta  = paired_delta(Q_LBL, Q_NL).rename(columns={'mean_delta': 'Qwen'})
llava_delta = paired_delta(L_LBL, L_NL).rename(columns={'mean_delta': 'LLaVA'})
pd.concat([qwen_delta[['Qwen']], llava_delta[['LLaVA']]], axis=1).round(3)

,Qwen,LLaVA
label,,
ball,0.006,0.475
circle,-0.065,0.173
planet,0.006,0.244


Reading: Qwen circle = −0.065 (suppression); LLaVA all positive
(ball +0.475 dominates). The H2 reframing in M4b was Qwen-specific.

## Visual-saturation hypothesis — `PMR(_nolabel)` per object_level

In [3]:
qwen_nl_obj  = Q_NL.groupby('object_level').pmr.mean().rename('Qwen')
llava_nl_obj = L_NL.groupby('object_level').pmr.mean().rename('LLaVA')
pd.concat([qwen_nl_obj, llava_nl_obj], axis=1).round(3)

,Qwen,LLaVA
object_level,,
filled,0.933,0.317
line,0.942,0.142
shaded,0.942,0.592
textured,0.975,0.483


Qwen's no-label PMR sits at 0.93–0.98 across the abstraction axis —
no headroom for a positive label contribution. LLaVA sits at 0.14–0.59
and the language prior has full visibility over an unsaturated dynamic
range.

## H1 — labeled S-curve (mean over labels, open prompt)

In [4]:
qwen_obj  = Q_LBL.groupby('object_level').pmr.mean().rename('Qwen labeled')
llava_obj = L_LBL.groupby('object_level').pmr.mean().rename('LLaVA labeled')
pd.concat([qwen_obj, qwen_nl_obj.rename('Qwen no-label'),
           llava_obj, llava_nl_obj.rename('LLaVA no-label')], axis=1).round(3)

,Qwen labeled,Qwen no-label,LLaVA labeled,LLaVA no-label
object_level,,,,
filled,0.933,0.933,0.656,0.317
line,0.906,0.942,0.508,0.142
shaded,0.933,0.942,0.753,0.592
textured,0.950,0.975,0.806,0.483


LLaVA labeled gives the cleanest monotone S-curve (0.51 → 0.81) seen
in this study. Qwen labeled is at ceiling (0.91–0.95).

## H7 — GAR by label (cross-model)

In [5]:
qwen_gar  = Q_LBL.groupby('label').gar.mean().rename('Qwen')
llava_gar = L_LBL.groupby('label').gar.mean().rename('LLaVA')
pd.concat([qwen_gar, llava_gar], axis=1).round(3)

,Qwen,LLaVA
label,,
ball,0.706,0.356
circle,0.753,0.153
planet,0.319,0.072


`planet GAR < ball/circle GAR` holds in both models — orbital-routing
is preserved. Magnitudes differ but direction is the same.

## Cleanest cell — `line/blank/none` × label, side-by-side

In [6]:
def cell_table(labeled, label_free):
    cell_filter = lambda d: d[(d.object_level == 'line') & (d.bg_level == 'blank') & (d.cue_level == 'none')]
    rows = [('_nolabel', cell_filter(label_free))]
    for lab in ['ball', 'circle', 'planet']:
        rows.append((lab, cell_filter(labeled[labeled.label == lab])))
    return pd.DataFrame([
        {'label': name, 'pmr': sub.pmr.mean(), 'hold_still': sub.hold_still.mean(), 'n': len(sub)}
        for name, sub in rows
    ]).set_index('label')

print('Qwen on line/blank/none:')
print(cell_table(Q_LBL, Q_NL).round(2))
print()
print('LLaVA on line/blank/none:')
print(cell_table(L_LBL, L_NL).round(2))

Qwen on line/blank/none:
          pmr  hold_still   n
label                        
_nolabel  0.4         0.2  10
ball      0.4         0.6  10
circle    0.1         1.0  10
planet    0.7         0.3  10

LLaVA on line/blank/none:
          pmr  hold_still   n
label                        
_nolabel  0.0         0.0  10
ball      0.9         0.0  10
circle    0.0         0.0  10
planet    0.5         0.0  10


On the fully abstract `line/blank/none` cell:
- Qwen: `_nolabel` PMR=0.40 (visual ambiguity → mixed); `ball` keeps PMR same but shifts regime to static; `circle` suppresses to 0.10; `planet` adds orbit prior to reach 0.70.
- LLaVA: `_nolabel` PMR=0; `ball` raises to 0.9; `circle` stays at 0; `planet` reaches 0.5. The language prior carries all the activation work; without it, LLaVA defaults to descriptive geometric responses.

## Sample LLaVA responses (T=0.7) on the fork-revealing cell

In [7]:
def sample_responses(df, where, n=3):
    sub = df[(df.object_level == where[0]) & (df.bg_level == where[1]) & (df.cue_level == where[2])]
    return [r.raw_text[:140] for _, r in sub.head(n).iterrows()]

print('LLaVA × line/blank/none × ball:')
for t in sample_responses(L_LBL[L_LBL.label == 'ball'], ('line', 'blank', 'none')):
    print(f'  {t!r}')
print('\nLLaVA × line/blank/none × circle:')
for t in sample_responses(L_LBL[L_LBL.label == 'circle'], ('line', 'blank', 'none')):
    print(f'  {t!r}')
print('\nLLaVA × line/blank/none × planet:')
for t in sample_responses(L_LBL[L_LBL.label == 'planet'], ('line', 'blank', 'none')):
    print(f'  {t!r}')
print('\nLLaVA × line/blank/none × _nolabel:')
for t in sample_responses(L_NL, ('line', 'blank', 'none')):
    print(f'  {t!r}')

LLaVA × line/blank/none × ball:
  'The ball will roll off the edge of the circle.'
  'The ball will fall.'
  'The ball will fall into a hole.'

LLaVA × line/blank/none × circle:
  'The circle will become more visible.'
  'The circle is going to get smaller and smaller.'
  'The circle will become a dot.'

LLaVA × line/blank/none × planet:
  'In the next moment, the planet will continue to spin and orbit around the sun.'
  'The planet will be consumed by a black hole.'
  'The planet will be consumed by a black hole.'

LLaVA × line/blank/none × _nolabel:
  'A round object is in the middle of the image, and the next part of the image is white.'
  'The circle is located at the center of the image. The next state or motion would be the circle expanding or contracting, depending on the co'
  'A black circle is seen in the background and the center of the circle is white.'
